# Causal IQ-Learn on PointMaze Medium

In [1]:
import random
import copy
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import ContinuousActor
from causal_rl.algo.imitation.gail.causal_gail import *
from causal_rl.algo.imitation.iqlearn.core_net import IQLearnQNetwork
from causal_rl.algo.imitation.iqlearn.causal_iqlearn import (
    IQLearnReplayBuffer, iq_init_expert_buffer,
    rollout_iqlearn_episode, iqlearn_update_critic, iqlearn_update_actor,
    soft_update, evaluate_iqlearn_policy,
)

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '6'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 1
hidden_dims = {'L'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, L hidden
train_env = PointMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, L hidden
eval_env = PointMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = PointMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'P0', 'P1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_pointmed.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 577619 trajectories


In [8]:
dims = {
    'P': 2,
    # 'L': 2,
    'W': 2,
    'X': 2
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window
causal_Z_trim = trim_Z_sets(Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags
causal_encode, causal_z_dim, causal_slots = build_windowed_z_encoder(
    causal_Z_trim,
    dims=dims,
    lookback=lookback,
)

encode = causal_encode
z_dim = causal_z_dim
Z_trim = causal_Z_trim
causal_z_dim

6

## Hyperparameters

In [10]:
# Shared SAC hyperparameters
total_timesteps = 2_000_000
batch_size = 256
gamma = 0.99
tau = 0.005
actor_lr = 3e-4
critic_lr = 3e-4
alpha_lr = 3e-4
hidden_dim = 256
buffer_capacity = 1_000_000
expert_capacity_ratio = 0.5
start_steps = 5_000
log_every = 50
eval_episodes = 10
max_grad_norm = 1.0
utd_ratio = 0.25  # update-to-data ratio: 1 gradient update per 4 env steps

# Actor architecture (match GAIL)
num_blocks_actor = 3
dropout_actor = 0.05
layernorm_actor = True

# IQ-Learn specific
num_v_samples = 16

# Environment action space
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())
target_entropy = -float(action_dim)

## Network Initialization

In [11]:
actor = ContinuousActor(
    num_inputs=z_dim, num_outputs=action_dim,
    hidden_size=hidden_dim, std=0.0,
    action_low=action_low, action_high=action_high,
    num_blocks=num_blocks_actor, dropout=dropout_actor, layernorm=layernorm_actor,
).to(device)

q1 = IQLearnQNetwork(z_dim, action_dim, hidden_dim,
                      num_blocks=num_blocks_actor, dropout=dropout_actor,
                      layernorm=layernorm_actor).to(device)
q2 = IQLearnQNetwork(z_dim, action_dim, hidden_dim,
                      num_blocks=num_blocks_actor, dropout=dropout_actor,
                      layernorm=layernorm_actor).to(device)
tq1 = copy.deepcopy(q1)
tq2 = copy.deepcopy(q2)
for p in tq1.parameters(): p.requires_grad = False
for p in tq2.parameters(): p.requires_grad = False

actor_optim = torch.optim.Adam(actor.parameters(), lr=actor_lr)
q1_optim = torch.optim.Adam(q1.parameters(), lr=critic_lr)
q2_optim = torch.optim.Adam(q2.parameters(), lr=critic_lr)

# Cosine LR schedule for critics
estimated_total_updates = int(total_timesteps * utd_ratio)
q1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q1_optim, T_max=estimated_total_updates)
q2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(q2_optim, T_max=estimated_total_updates)

# Automatic entropy tuning
log_alpha = torch.zeros(1, requires_grad=True, device=device)
alpha_optim = torch.optim.Adam([log_alpha], lr=alpha_lr)

buffer = IQLearnReplayBuffer(buffer_capacity, expert_capacity_ratio)
iq_init_expert_buffer(records, encode, buffer, device)

Expert buffer: 500000 transitions from 1000 episodes


## Training

In [12]:
best_eval = -float('inf')
best_state_dict = copy.deepcopy(actor.state_dict())

ts = 0
ep = 0
logs = []

while ts < total_timesteps:
    ep_data = rollout_iqlearn_episode(
        train_env, actor, buffer, encode,
        num_steps, device, deterministic=False, seed=seed + 20000 + ep
    )
    ts += ep_data['episode_length']
    ep += 1

    if ts > start_steps and len(buffer.policy_buffer) >= batch_size // 2:
        n_updates = max(1, int(ep_data['episode_length'] * utd_ratio))
        for _ in range(n_updates):
            alpha_val = log_alpha.exp().item()
            iqlearn_update_critic(
                q1, q2, tq1, tq2, actor, alpha_val, buffer,
                batch_size, gamma, q1_optim, q2_optim,
                device, num_v_samples, max_grad_norm,
            )
            iqlearn_update_actor(
                actor, q1, q2, log_alpha, target_entropy,
                actor_optim, alpha_optim,
                buffer, batch_size, device, max_grad_norm,
            )
            soft_update(q1, tq1, tau)
            soft_update(q2, tq2, tau)

            q1_scheduler.step()
            q2_scheduler.step()

            # Alpha clamping (IQ-Learn stability fix)
            with torch.no_grad():
                log_alpha.clamp_(min=np.log(0.001), max=np.log(0.1))

    if ep % log_every == 0:
        eval_ret = evaluate_iqlearn_policy(
            train_env, actor, encode, num_steps, device, eval_episodes, seed=42
        )
        logs.append({
            'episode': ep, 'timesteps': ts,
            'eval_return': eval_ret, 'train_return': ep_data['episode_return'],
            'alpha': log_alpha.exp().item(),
        })
        print(
            f"[Causal IQ-Learn ep {ep}] "
            f"ts={ts}, eval={eval_ret:.2f}, "
            f"train={ep_data['episode_return']:.2f}, "
            f"alpha={log_alpha.exp().item():.4f}"
        )

        # Best checkpoint tracking
        if eval_ret > best_eval:
            best_eval = eval_ret
            best_state_dict = copy.deepcopy(actor.state_dict())

# Restore best
actor.load_state_dict(best_state_dict)
print(f"Restored best checkpoint with eval={best_eval:.2f}")

[Causal IQ-Learn ep 50] ts=18169, eval=-57.19, train=-36.85, alpha=0.0648


[Causal IQ-Learn ep 100] ts=49657, eval=-52.22, train=-53.99, alpha=0.0371


[Causal IQ-Learn ep 150] ts=78463, eval=-749.49, train=-620.90, alpha=0.0298


[Causal IQ-Learn ep 200] ts=122755, eval=-741.58, train=-391.29, alpha=0.1000


[Causal IQ-Learn ep 250] ts=166421, eval=-741.78, train=-894.35, alpha=0.1000


...

[Causal IQ-Learn ep 2800] ts=1802005, eval=-740.89, train=-1012.13, alpha=0.0216


[Causal IQ-Learn ep 2850] ts=1852005, eval=-732.02, train=-934.90, alpha=0.0199


[Causal IQ-Learn ep 2900] ts=1902005, eval=-695.72, train=-484.64, alpha=0.0204


[Causal IQ-Learn ep 2950] ts=1952005, eval=-717.22, train=-499.08, alpha=0.0152


Restored best checkpoint with eval=-50.02


## Evaluation

In [13]:
causal_iqlearn_policy = make_gail_policy(actor, encode, device=device, deterministic=True)
causal_iqlearn_policies = make_shared_policy_dict(causal_iqlearn_policy)

In [14]:
num_eval_eps = 100
causal_iqlearn_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_iqlearn_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(causal_iqlearn_returns)

Starting episode 1/100...


  Episode 1 ended at step 105 (terminated: True, truncated: False).
Starting episode 2/100...
  Episode 2 ended at step 110 (terminated: True, truncated: False).
Starting episode 3/100...


  Episode 3 ended at step 105 (terminated: True, truncated: False).
Starting episode 4/100...
  Episode 4 ended at step 106 (terminated: True, truncated: False).
Starting episode 5/100...
  Episode 5 ended at step 108 (terminated: True, truncated: False).
Starting episode 6/100...


  Episode 6 ended at step 107 (terminated: True, truncated: False).
Starting episode 7/100...
  Episode 7 ended at step 112 (terminated: True, truncated: False).
Starting episode 8/100...


  Episode 8 ended at step 107 (terminated: True, truncated: False).
Starting episode 9/100...
  Episode 9 ended at step 106 (terminated: True, truncated: False).
Starting episode 10/100...


...

  Episode 91 ended at step 103 (terminated: True, truncated: False).
Starting episode 92/100...
  Episode 92 ended at step 105 (terminated: True, truncated: False).
Starting episode 93/100...
  Episode 93 ended at step 111 (terminated: True, truncated: False).
Starting episode 94/100...


  Episode 94 ended at step 108 (terminated: True, truncated: False).
Starting episode 95/100...
  Episode 95 ended at step 109 (terminated: True, truncated: False).
Starting episode 96/100...
  Episode 96 ended at step 108 (terminated: True, truncated: False).
Starting episode 97/100...


  Episode 97 ended at step 105 (terminated: True, truncated: False).
Starting episode 98/100...
  Episode 98 ended at step 104 (terminated: True, truncated: False).
Starting episode 99/100...
  Episode 99 ended at step 108 (terminated: True, truncated: False).
Starting episode 100/100...


  Episode 100 ended at step 111 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


10807

In [15]:
causal_iqlearn_episode_rewards = defaultdict(float)
for rec in causal_iqlearn_returns:
    ep = rec['episode']
    causal_iqlearn_episode_rewards[ep] += float(rec['reward'])

causal_iqlearn_rewards = [causal_iqlearn_episode_rewards[e] for e in range(num_eval_eps)]
sum(causal_iqlearn_rewards) / num_eval_eps

-60.544772967252065

## Save Model

In [16]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'ciqlearn_pointmed.pt')

ckpt = {
    'state_dict': actor.state_dict(),
    'z_dim': causal_z_dim,
    'action_dim': action_dim,
    'hidden_size_actor': hidden_dim,
    'num_blocks_actor': num_blocks_actor,
    'dropout_actor': dropout_actor,
    'layernorm_actor': layernorm_actor,
    'final_tanh': True,
    'action_bounds_low': eval_env.env.action_space.low,
    'action_bounds_high': eval_env.env.action_space.high,
    'Z_sets': causal_Z_trim,
    'dims': dims,
    'lookback': lookback,
}

torch.save(ckpt, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/ciqlearn_pointmed.pt
